# **Collaborative Filtering Recommendation System**

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Read Book CSV

In [3]:
books = pd.read_csv("data/BX-Books.csv", sep=";", on_bad_lines='skip', encoding="latin-1", low_memory=False)
print(books.shape)
books.head(3)

(271360, 8)


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...


In [4]:
books.columns

Index(['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher',
       'Image-URL-S', 'Image-URL-M', 'Image-URL-L'],
      dtype='object')

#### Choose needed columns

In [5]:
books = books[['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher', 'Image-URL-L']]
books.head(3)

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...


#### Columns renaming

In [6]:
books.rename(columns = {
    "ISBN": "isbn",
    "Book-Title": "title",
    "Book-Author": "author",
    "Year-Of-Publication": "year",
    "Publisher": "publisher",
    "Image-URL-L": "img_url"
}, inplace=True)

books.head(3)

,isbn,title,author,year,publisher,img_url
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...


### Read User CSV

In [7]:
users = pd.read_csv("data/BX-Users.csv", sep=";", on_bad_lines='skip', encoding="latin-1", low_memory=False)
print(users.shape)
users.head(3)

(278858, 3)


,User-ID,Location,Age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN


In [8]:
users.rename(columns = {
    "User-ID": "user_id",
    "Location": "location",
    "Age": "age"
}, inplace=True)

users.head(3)

,user_id,location,age
0,1,"nyc, new york, usa",NaN
1,2,"stockton, california, usa",18.0
2,3,"moscow, yukon territory, russia",NaN


### Read Ratings CSV

In [9]:
ratings = pd.read_csv("data/BX-Book-Ratings.csv", sep=";", on_bad_lines='skip', encoding="latin-1", low_memory=False)
print(ratings.shape)
ratings.sample(3) # Book-Rating means how many start rate not count

(1149780, 3)


,User-ID,ISBN,Book-Rating
653051,158262,0446603589,0
1024772,245827,0140017895,0
716532,173632,3423030232,0


In [10]:
ratings.columns

Index(['User-ID', 'ISBN', 'Book-Rating'], dtype='object')

In [11]:
ratings.rename(columns = {
    "User-ID": "user_id",
    "ISBN": "isbn",
    "Book-Rating": "ratings"
}, inplace=True)

ratings.head(3)

,user_id,isbn,ratings
0,276725,034545104X,0
1,276726,0155061224,5
2,276727,0446520802,0


In [12]:
# value_counts >> How many times EACH user appears, original tbale may have no of records with same user_id, her we count ho many are there
ratings['user_id'].value_counts() # user 11676 has rated 13602 times/books

user_id
11676     13602
198711     7550
153662     6109
98391      5891
35859      5850
          ...  
119573        1
276706        1
276697        1
276679        1
276676        1
Name: count, Length: 105283, dtype: int64

In [13]:
ratings['user_id'].unique().shape # No of unique users

(105283,)

In [14]:
# Converts counts into True/False
# x is a boolean Series, Index = user_id    (mask)
x = ratings['user_id'].value_counts() > 200
x

user_id
11676      True
198711     True
153662     True
98391      True
35859      True
          ...  
119573    False
276706    False
276697    False
276679    False
276676    False
Name: count, Length: 105283, dtype: bool

In [15]:
# x[...] means filtering
# When you pass a boolean mask inside [], Pandas keeps only True values
# x.sum() also correct, .sum() directly counts True values

x[x].shape # x[x] > filter only True values

(899,)

In [16]:
x.sum()

np.int64(899)

In [17]:
y = x[x].index # gives only the user IDs (names), not True/False
y # list of users who have more than 200 ratings

Index([ 11676, 198711, 153662,  98391,  35859, 212898, 278418,  76352, 110973,
       235105,
       ...
       116122,  44296,  28634,  59727,  73681, 274808, 188951,   9856, 155916,
       268622],
      dtype='int64', name='user_id', length=899)

In [32]:
# we fetch users who rated more than 200 times as a list 'y', then we crate a boolean mask using '.isin()', then apply that to entire table
# ratings['user_id'].isin(y) >>> return True / False for each row
# finally we have a table 'filtered_ratings' only keeps users who are in y

filtered_ratings = ratings[ratings['user_id'].isin(y)]
print(filtered_ratings.shape)
filtered_ratings.head()

(526356, 3)


,user_id,isbn,ratings
1456,277427,002542730X,10
1457,277427,0026217457,0
1458,277427,003008685X,8
1459,277427,0030615321,0
1460,277427,0060002050,0


In [28]:
print(books.shape)
books.head(2)

(271360, 6)


,isbn,title,author,year,publisher,img_url
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...


#### Merge book with ratings

In [24]:
# Keep only rows where isbn exists in BOTH tables when merging
rating_with_books = filtered_ratings.merge(books, on = "isbn")

In [39]:
print(rating_with_books.shape)
rating_with_books.sample(5) # Each row = one user rating one book

(487671, 8)


,user_id,isbn,ratings,title,author,year,publisher,img_url
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...
1,277427,0026217457,0,Vegetarian Times Complete Cookbook,Lucy Moll,1995,John Wiley &amp; Sons,http://images.amazon.com/images/P/0026217457.0...
2,277427,003008685X,8,Pioneers,James Fenimore Cooper,1974,Thomson Learning,http://images.amazon.com/images/P/003008685X.0...
3,277427,0030615321,0,"Ask for May, Settle for June (A Doonesbury book)",G. B. Trudeau,1982,Henry Holt &amp; Co,http://images.amazon.com/images/P/0030615321.0...
4,277427,0060002050,0,On a Wicked Dawn (Cynster Novels),Stephanie Laurens,2002,Avon Books,http://images.amazon.com/images/P/0060002050.0...


### How many times each book got raitings

In [44]:
# groupby('title') >>> "Put all rows of the same book together”
# Now we count how many ratings each book has, we count number of rows in each group
# .reset_index() >>> After groupby, Pandas makes title the index (not a normal column), We convert it back into a normal column
num_ratings = rating_with_books.groupby('title')['ratings'].count().reset_index()

In [48]:
num_ratings.sample(5) # How many people rated each book

,title,ratings
126289,The Hallowed Isle : The Book Of The Sword And ...,1
50802,"Greek Myths (Hotshots Series , No 10)",1
124527,The Forgotten War (Star Trek: The Next Generat...,3
115745,"The Anglo-Saxon Age, C.400-1042 (History of En...",1
148803,"Tuscan Tycoon's Wife (Larger Print, 606)",1


In [53]:
num_ratings.rename(columns={'ratings' : 'num_of_ratings'}, inplace=True)

In [54]:
num_ratings.head()

,title,num_of_ratings
0,A Light in the Storm: The Civil War Diary of ...,2
1,Always Have Popsicles,1
2,Apple Magic (The Collector's series),1
3,Beyond IBM: Leadership Marketing and Finance ...,1
4,Clifford Visita El Hospital (Clifford El Gran...,1


In [55]:
final_ratings = rating_with_books.merge(num_ratings, on="title")

In [62]:
# for each row ( for each book) we have a num_of_ratings value, since rating_with_books had same in book servral rows each somtimes same 
# num_of_ratings value can be several palces bcs user_id + title both together unique eg :
# 277427 002542730X 10 Politically Correct.. James Finn.. ... 82
# 52614  002542730X 5  Politically Correct.. James Finn.. ... 82

final_ratings.head(5) 

,user_id,isbn,ratings,title,author,year,publisher,img_url,num_of_ratings
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,http://images.amazon.com/images/P/002542730X.0...,82
1,277427,0026217457,0,Vegetarian Times Complete Cookbook,Lucy Moll,1995,John Wiley &amp; Sons,http://images.amazon.com/images/P/0026217457.0...,7
2,277427,003008685X,8,Pioneers,James Fenimore Cooper,1974,Thomson Learning,http://images.amazon.com/images/P/003008685X.0...,1
3,277427,0030615321,0,"Ask for May, Settle for June (A Doonesbury book)",G. B. Trudeau,1982,Henry Holt &amp; Co,http://images.amazon.com/images/P/0030615321.0...,1
4,277427,0060002050,0,On a Wicked Dawn (Cynster Novels),Stephanie Laurens,2002,Avon Books,http://images.amazon.com/images/P/0060002050.0...,13


In [65]:
mask = final_ratings['num_of_ratings'] >= 50
final_ratings = final_ratings[mask]

In [69]:
print(final_ratings.shape)
final_ratings.sample(5)

(61853, 9)


,user_id,isbn,ratings,title,author,year,publisher,img_url,num_of_ratings
137883,78783,0743437640,0,The Mulberry Tree,Jude Deveraux,2003,Pocket,http://images.amazon.com/images/P/0743437640.0...,69
325168,189334,0671014919,0,My Gal Sunday,Mary Higgins Clark,2003,Pocket,http://images.amazon.com/images/P/0671014919.0...,70
153182,88733,043936213X,9,Harry Potter and the Sorcerer's Stone (Book 1),J. K. Rowling,2001,Scholastic,http://images.amazon.com/images/P/043936213X.0...,78
85943,43806,0671888587,0,I'll Be Seeing You,Mary Higgins Clark,1994,Pocket,http://images.amazon.com/images/P/0671888587.0...,77
81851,40889,0316666343,0,The Lovely Bones: A Novel,Alice Sebold,2002,"Little, Brown",http://images.amazon.com/images/P/0316666343.0...,270


In [71]:
# One user should have only one rating per book
final_ratings.drop_duplicates(['user_id', 'title'], inplace=True)

C:\Users\Lapmart\AppData\Local\Temp\ipykernel_21052\167040141.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_ratings.drop_duplicates(['user_id', 'title'], inplace=True)


In [72]:
final_ratings.shape

(59850, 9)

# Pivot table
- this will contain user_ids with book title with ratings

In [75]:
book_pivot = final_ratings.pivot_table(columns='user_id', index='title', values='ratings')

In [82]:
book_pivot

user_id,254,2276,2766,2977,3363,3757,4017,4385,6242,6251,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
84 Charing Cross Road,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Year of Wonders,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,7.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
You Belong To Me,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Zen and the Art of Motorcycle Maintenance: An Inquiry into Values,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [83]:
book_pivot.shape

(742, 888)

In [84]:
book_pivot.fillna(0, inplace=True)

### We have many 0s whewn we cal distnace 0-0=0 not needed, let use scipy to avoid that

In [85]:
from scipy.sparse import csr_matrix

In [86]:
book_sparse = csr_matrix(book_pivot)
book_sparse

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 14961 stored elements and shape (742, 888)>

### For clustering we use K-Nearest-Neighbors algorithm

In [95]:
from sklearn.neighbors import NearestNeighbors

# There are different ways to find nearest neighbors, brute = Brute Force search
# Compare with every single book to find the closest ones, It calculates distance like: >>> distance(bookA, bookB) >>> for ALL books
model = NearestNeighbors(algorithm='brute')

In [98]:
model.fit(book_sparse)

NearestNeighbors(algorithm='brute')

In [99]:
# smaple data
sample_data = book_pivot.iloc[237, :] # Entire rcord for book index 237, 238th record in book_pivot table
sample_data

user_id
254       9.0
2276      0.0
2766      0.0
2977      0.0
3363      0.0
         ... 
275970    9.0
277427    0.0
277478    0.0
277639    0.0
278418    0.0
Name: Harry Potter and the Chamber of Secrets (Book 2), Length: 888, dtype: float64

In [100]:
distance, suggessions = model.kneighbors(sample_data.values.reshape(1, -1), n_neighbors=6)

In [101]:
# distance to each cluster
distance # 0 means distence to itself

array([[ 0.        , 67.75691847, 68.05145112, 72.277244  , 75.81556568,
        76.30203143]])

In [102]:
suggessions

array([[237, 238, 240, 241, 184, 536]])

In [103]:
for i in range(len(suggessions)):
    print(book_pivot.index[suggessions[i]])

Index(['Harry Potter and the Chamber of Secrets (Book 2)',
       'Harry Potter and the Goblet of Fire (Book 4)',
       'Harry Potter and the Prisoner of Azkaban (Book 3)',
       'Harry Potter and the Sorcerer's Stone (Book 1)', 'Exclusive',
       'The Cradle Will Fall'],
      dtype='object', name='title')


In [104]:
book_pivot.index[3]

'4 Blondes'

In [105]:
book_names = book_pivot.index

In [111]:
import pickle
pickle.dump(model, open('artifacts/model.pkl', 'wb'))
pickle.dump(book_names, open('artifacts/book_names.pkl', 'wb'))
pickle.dump(final_ratings, open('artifacts/final_ratings.pkl', 'wb'))
pickle.dump(book_pivot, open('artifacts/book_pivot.pkl', 'wb'))

##### suggession returns book index from book_pivot table >>> we get book title from it >>> we can get all data of the book using that title

In [113]:
print(np.where(book_pivot.index == "2nd Chance"))
print(np.where(book_pivot.index == "2nd Chance")[0])
print(np.where(book_pivot.index == "2nd Chance")[0][0])

(array([2]),)
[2]
2


In [116]:
def recoomend_book(book_name):
    book_id  = np.where(book_pivot.index == book_name)[0][0]
    distance, suggessions = model.kneighbors(book_pivot.iloc[book_id, : ].values.reshape(1, -1), n_neighbors=6)

    for i in range(len(suggessions)):
        books = book_pivot.index[suggessions[i]]
        for j in books:
            print(j)

In [121]:
book_name = "A Bend in the Road"

# This model work as follows
# if some one open a book A model look for books that in same cluster with min distance. The cluster is created using ratings of each user.
# If users read/rated book A, Book B and also Book C, then another user open/click on Book A, then we can say/ suggest some 
# other books that in same cluster with Book A with min distance to Book A, cluster creation done by actualy according to user interactions 
# here only ratings in content based system we use genre, title, category, ... but in collaborative approach we use user interaction to cluster 
# simillar vibe books

recoomend_book(book_name)

A Bend in the Road
Exclusive
The Cradle Will Fall
No Safe Place
Family Album
Lake Wobegon days
